In [4]:
pip install requests pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import time
from datetime import datetime, timezone, timedelta

import requests
import pandas as pd

BASE_URL = "https://api.binance.com"
KLINES_ENDPOINT = "/api/v3/klines"

INTERVAL = "1h" # for 5 min interval, change to "5min"
INTERVAL_MS = 60 * 60 * 1000  # 1 hour in ms, but for 5min interval change again

COLUMNS = [
    "open_time_ms",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time_ms",
    "quote_asset_volume",
    "num_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore",
]

def dt_to_ms(dt: datetime) -> int:
    return int(dt.timestamp() * 1000)

def fetch_klines(symbol: str, start_ms: int, end_ms: int, limit: int = 1000):
    url = BASE_URL + KLINES_ENDPOINT
    params = {
        "symbol": symbol,
        "interval": INTERVAL,
        "startTime": start_ms,
        "endTime": end_ms,
        "limit": limit,  # max 1000 on this endpoint :contentReference[oaicite:1]{index=1}
    }
    r = requests.get(url, params=params, timeout=30)
    # Handle rate limiting politely
    if r.status_code in (418, 429):
        # Binance can temporarily block/limit; wait and retry
        retry_after = int(r.headers.get("Retry-After", "5"))
        time.sleep(max(5, retry_after)) #might need to inc throttle time incase of high freq because of higher number of API calls
        r = requests.get(url, params=params, timeout=30)

    r.raise_for_status()
    return r.json()

def download_hourly_btc( #can rename to download_fivemin_btc
    symbol: str = "BTCUSDT",
    start_date: str = "2021-01-01",
    end_date: str = "2025-12-31",
    out_csv: str = "btc_hourly_binance_2021_2025.csv",
):
    # Binance interprets startTime/endTime in UTC :contentReference[oaicite:2]{index=2}
    start_dt = datetime.fromisoformat(start_date).replace(tzinfo=timezone.utc)
    # make end exclusive by adding 1 day, then subtract 1ms for inclusive endTime
    end_dt_exclusive = datetime.fromisoformat(end_date).replace(tzinfo=timezone.utc) + timedelta(days=1)

    start_ms = dt_to_ms(start_dt)
    end_ms = dt_to_ms(end_dt_exclusive) - 1

    all_rows = []
    cur_ms = start_ms

    while cur_ms <= end_ms:
        batch = fetch_klines(symbol, cur_ms, end_ms, limit=1000)
        if not batch:
            break

        all_rows.extend(batch)

        last_open_ms = batch[-1][0]
        next_ms = last_open_ms + INTERVAL_MS

        # Safety: prevent infinite loops
        if next_ms <= cur_ms:
            break

        cur_ms = next_ms

        # Light throttle (rate limits are weight-based; this keeps things smooth) :contentReference[oaicite:3]{index=3}
        time.sleep(0.05)

    df = pd.DataFrame(all_rows, columns=COLUMNS)

    # Convert types
    numeric_cols = ["open", "high", "low", "close", "volume", "quote_asset_volume",
                    "taker_buy_base_volume", "taker_buy_quote_volume"]
    df[numeric_cols] = df[numeric_cols].astype(float)
    df["num_trades"] = df["num_trades"].astype(int)

    # Human-readable timestamps
    df["open_time_utc"] = pd.to_datetime(df["open_time_ms"], unit="ms", utc=True)
    df["close_time_utc"] = pd.to_datetime(df["close_time_ms"], unit="ms", utc=True)

    # If you mean “hourly price” as the hourly close:
    df["hourly_price_close"] = df["close"]

    # De-dup + sort just in case
    df = df.drop_duplicates(subset=["open_time_ms"]).sort_values("open_time_ms").reset_index(drop=True)

    df.to_csv(out_csv, index=False)
    print(f"Saved {len(df):,} rows to {out_csv}")

if __name__ == "__main__":
    download_hourly_btc() # and this to download_fivemin_btc


/Users/paridhiagarwal/Library/Python/3.8/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Saved 43,810 rows to btc_hourly_binance_2021_2025.csv
